# Retrieval-guided master-prompt selection

## 1. Goal and single-split contract

This notebook compares configured master prompts while holding retrieval and inference controls explicit. One run evaluates exactly one configured split:

- `validation` maps to the Bias-in-Bios `dev` source;
- `test` maps to the Bias-in-Bios `test` source;
- the unselected split may appear in descriptive source counts, but it is never embedded, predicted, scored, ranked, or plotted.

Every configured condition is evaluated on the selected split. Conditions are ranked separately within each language model, and exactly one row per language model receives `is_best=True`. There is no automatic transition from validation to test.

## 2. Environment and imports

Run this notebook from the repository root in the `prompt-selection` Conda environment. Imports below use the same modules as `app.py`; there are no standalone dataset downloads or direct model probes.

In [1]:
# from modeling import load_language_model, clear_language_model_memory
#
# ass = ['Qwen/Qwen3.6-27B', 'Qwen/Qwen3.5-27B', 'Qwen/Qwen3.6-35B-A3B', 'google/gemma-4-31B-it']
# bss = ['6a9e13bd6fc8f0983b9b99948120bc37f49c13e9', 'fc05daec18b0a78c049392ed2e771dde82bdf654',
#      '995ad96eacd98c81ed38be0c5b274b04031597b0', '842da3794eaa0b77d5f08bae87a17459d91ff475']
# c = 'bfloat16'
#
# for a, b in zip(ass, bss):
#     tokenizer, model = load_language_model(a, b, 'mps', c)
#     print(a, tokenizer.model_max_length)
#
#     del tokenizer, model
#     clear_language_model_memory('mps')

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Qwen/Qwen3.6-27B 262144


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

...

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/693 [00:00<?, ?it/s]

Qwen/Qwen3.6-35B-A3B 262144


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

google/gemma-4-31B-it 1000000000000000019884624838656


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, JSON, Markdown, display

from dataset import (
    calculate_dataset_counts,
    load_data,
    select_run_data,
    task_settings,
)
from evaluation import resolve_metric_column
from modeling import build_prompt
from configuration import load_config, validate_config
from pipeline import run_experiment

PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError('Open this notebook from the repository root.')

## 3. Load and validate the configuration

`config.yaml` is the single source of experiment settings. Validation rejects unknown enums, invalid scalar types, empty values, duplicates, unsupported prompt placeholders, and inconsistent pool sizes before model work begins.

In [ ]:
config = load_config(CONFIG_PATH)
validate_config(config)

evaluation_split = config['defaults']['evaluation_split']
target, audit_column, professions, labels = task_settings(config)

display(Markdown(
    f'**Configured run:** hold out `{target}` and evaluate `{evaluation_split}`; '
    f'the language model receives `hard_text + {audit_column}`.'
))

## 4. Condition count and runtime controls

Conditions per language model are the product of retrieval methods, embedding models, example counts, example orders, and prompt templates. Total predictions also multiply by language-model count and selected evaluation rows.

The main row multiplier is `evaluation_per_profession_gender`: selected rows equal `number of professions × 2 genders × evaluation_per_profession_gender`. Embedding sequence limits and loaded language-model context limits are checked; inputs fail instead of being silently truncated.

In [ ]:
retrieval = config['retrieval']
conditions_per_model = (
        len(retrieval['methods'])
        * len(retrieval['embedding_models'])
        * len(retrieval['example_counts'])
        * len(retrieval['example_orders'])
        * len(config['prompt_templates'])
)
language_model_count = len(config['inference']['language_models'])
evaluation_row_count = (
        len(professions)
        * 2
        * config['dataset']['evaluation_per_profession_gender']
)

pd.DataFrame([{
    'language_models': language_model_count,
    'conditions_per_language_model': conditions_per_model,
    'total_conditions': language_model_count * conditions_per_model,
    'selected_evaluation_rows': evaluation_row_count,
    'total_label_predictions': language_model_count * conditions_per_model * evaluation_row_count,
}])

## 5. Load all sources, select run rows, and inspect composition

`load_data()` normalizes all profession-filtered source rows into the canonical keys `train`, `validation`, and `test`. `select_run_data()` is the only step that applies the train cap and chooses balanced evaluation cells.

The source table is descriptive. The run table is authoritative for rows used by retrieval and evaluation.

In [ ]:
source_splits = load_data(config, PROJECT_ROOT)
train_rows, evaluation_rows = select_run_data(config, source_splits)

source_dataset_counts = calculate_dataset_counts(config, source_splits)
run_dataset_counts = calculate_dataset_counts(
    config,
    {'train': train_rows, evaluation_split: evaluation_rows},
)

display(Markdown('### Full filtered source composition'))
display(source_dataset_counts)
display(Markdown('### Rows selected for this run'))
display(run_dataset_counts)

assert {row['split'] for row in evaluation_rows} == {evaluation_split}
assert set(run_dataset_counts['split']) == {'train', evaluation_split}

## 6. Preview the language-model input

This preview uses the first selected query and representative selected train rows to show the exact structured-message format. During the run, retrieval chooses condition-specific demonstrations; each actual message list is saved in the prediction table's `prompt` column.

In [ ]:
preview_template_name, preview_template = next(iter(config['prompt_templates'].items()))
preview_example_count = max(config['retrieval']['example_counts'])
preview_messages = build_prompt(
    evaluation_rows[0],
    train_rows[:preview_example_count],
    target,
    labels,
    preview_template,
)

display(Markdown(f'**Template:** `{preview_template_name}`'))
display(JSON(preview_messages, expanded=True))

## 7. Run the experiment

This is the expensive cell. It embeds only the selected train and evaluation rows, loads each language model once, calculates each condition's metrics immediately, ranks within that model, then writes split-prefixed artifacts.

In [ ]:
run = run_experiment(config, PROJECT_ROOT, progress=print)

assert run['evaluation_split'] == evaluation_split
assert set(run['predictions']['evaluation_split']) == {evaluation_split}
display(Markdown(f'Artifacts written to `{run["run_dir"]}`.'))

## 8. Inspect rankings and current-split winners

Ranks restart at 1 for each language model. `is_best` identifies one winner per model, chosen using the configured metric and direction on this run's split.

In [ ]:
metric_column = resolve_metric_column(config['defaults']['ranking_metric'])
ranking_columns = [
    'evaluation_split', 'language_model', 'rank', 'is_best', 'condition', metric_column,
]
ranking_columns = [column for column in ranking_columns if column in run['results'].columns]
display(run['results'][ranking_columns])

best_conditions = run['results'].loc[run['results']['is_best']].copy()
assert len(best_conditions) == len(config['inference']['language_models'])
display(Markdown('### One current-split winner per language model'))
display(best_conditions[ranking_columns])

## 9. Inspect detailed metrics and plots

CSV metric tables retain every condition. The bounded previews below focus on winning conditions. Summary plots rank all conditions; detailed class, group, fairness, coverage, and confusion plots focus on the winners.

In [ ]:
best_names = set(best_conditions['condition'])
for table_name in ('class_metrics', 'group_metrics', 'fairness_metrics', 'confusion_matrix'):
    table = run[table_name]
    display(Markdown(f'### {table_name.replace("_", " ").title()}'))
    display(table.loc[table['condition'].isin(best_names)].head(200))

display(Markdown('### Plot files'))
for plot_name, plot_path in run['plots'].items():
    display(Markdown(f'**{plot_name.replace("_", " ").title()}**'))
    display(Image(filename=str(plot_path)))

## 10. Inspect best prompts and bounded prediction previews

The text report contains only this split's winners and ranking score. Prediction previews show true label, audit group, chosen label, condition metadata, and allowed-label scores; the complete table remains in the split-prefixed CSV.

In [ ]:
display(Markdown(run['best_prompts'].read_text(encoding='utf-8')))

prediction_columns = [
    'evaluation_split', 'language_model', 'query_id', 'true_label',
    'audit_group', 'predicted_label', 'condition', 'label_scores',
]
best_predictions = run['predictions'].loc[
    run['predictions']['condition'].isin(best_names), prediction_columns,
]
display(best_predictions.head(200))

## 11. Metric reference and next-run guidance

For class $c$, $Precision=TP/(TP+FP)$, $Recall=TP/(TP+FN)$, and $F1=2TP/(2TP+FP+FN)$. Macro metrics average defined class rates; weighted metrics use true class support. In this single-label multiclass task:

$$Accuracy=MicroPrecision=MicroRecall=MicroF1=WeightedRecall$$

$$BalancedAccuracy=MacroRecall$$

For audit group $g$, $SR_{c,g}=(TP+FP)/N_g$. Demographic-parity difference is the range of $SR$ across groups; equal-opportunity difference is the TPR range; equalized-odds difference is the larger of TPR and FPR ranges; predictive-parity difference is the precision range. Undefined denominators become `NaN`, and a disparity requires at least two defined groups.

Use quality and fairness metrics together. Keep `evaluation_split: validation` while developing prompt candidates. Change it deliberately to `test` only when you intend to evaluate the complete condition grid on test. Profession and gender are separate experiments with the same prompt candidates and separate winners; do not mix their rankings.